# BSDS demo — hidden brain-state dynamics with `ffs_bsds`

This notebook fits **Bayesian Switching Dynamical Systems** (Taghia et al. 2018; Cai et al. 2024)
to ROI time series and walks through the outputs, with figures inline.

BSDS is a *switching factor-analysis* model: an HMM over a handful of discrete **brain states**,
each with its own low-rank+diagonal covariance (its **functional-connectivity** pattern), fit by
variational Bayes. It gives you, moment by moment, which latent state the brain is in, how states
transition, how long they dwell, and what network pattern defines each.

Here we use synthetic data with a known ground truth so you can see the model recover it. At the
end are notes on pointing it at real data (and the equivalent one-line `ffs_bsds` call).

# TODO

Get data - like the 100 runs - we need to see if we can see switching states when there are not so many tasks
Alternative - fewer ROIs and do denoising on this data
Alterntaive - the MVPA data- more runs, maybe we need more data to ID states?


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from fastfuncstuff.dynamics import plots
from fastfuncstuff.dynamics.bsds.model import fit_bsds
from fastfuncstuff.dynamics.states import compute_state_stats

%matplotlib inline

TR = 0.72  # seconds; overridden in the real-data cell if you set one

## 1. A synthetic switching dataset

Each **session** is a `(D, N)` array — `D` ROIs by `N` timepoints. We simulate `K` states, each
with a distinct mean and a low-rank+diagonal covariance, and a *sticky* transition matrix (states
persist, matching real brain dynamics). Feeding several sessions is exactly how you'd analyse a
**densely-sampled individual**: runs/sessions become the model's session list.

In [ ]:
def simulate(k=4, d=12, r=2, t=500, n_sessions=15, stay=0.96, seed=0):
    rng = np.random.default_rng(seed)
    means = rng.standard_normal((k, d)) * 4.0
    chols, covs = [], []
    for _ in range(k):
        w = rng.standard_normal((d, r)) * 0.7
        cov = w @ w.T + 0.3 * np.eye(d)
        covs.append(cov); chols.append(np.linalg.cholesky(cov))
    trans = np.full((k, k), (1 - stay) / (k - 1)); np.fill_diagonal(trans, stay)
    sessions, truth = [], []
    for _ in range(n_sessions):
        z = np.empty(t, int); z[0] = rng.integers(k)
        for i in range(1, t): z[i] = rng.choice(k, p=trans[z[i-1]])
        y = np.stack([means[z[i]] + chols[z[i]] @ rng.standard_normal(d) for i in range(t)], axis=1)
        sessions.append(torch.tensor(y, dtype=torch.float64)); truth.append(z)
    return sessions, truth, np.array(covs)

sessions, truth, true_covs = simulate()
print(f'{len(sessions)} sessions, D={sessions[0].shape[0]} ROIs, N={sessions[0].shape[1]} TRs each')

### 1b. Using your own data — drop it in here

Flip `USE_REAL_DATA = True` and point the paths at your runs. Everything below is unchanged —
it just consumes the `sessions` list (each a `(D, N)` ROI-by-time tensor). Two loaders are shown,
both reusing the same helpers the `ffs_bsds` CLI uses, so the notebook and CLI agree.

**Real data must be preprocessed** (per-run detrend + z-score) — the simulated path skips it
because the synthetic data is already clean. `preprocess_sessions` does exactly what the CLI does.

In [ ]:
USE_REAL_DATA = True
import glob
import os
import pickle

WORKDIR = '/mnt/belegost/Projects/TASKFORCE/derivatives/bsds'
os.makedirs(WORKDIR, exist_ok=True)


if USE_REAL_DATA:
    TR = 1.0
    from fastfuncstuff.dynamics.preprocess import preprocess_sessions

    # ---- choose your parcellation (sets the cache key) ----
    ATLAS = '/mnt/belegost/Projects/TASKFORCE/derivatives/atlas/resamp_aparc+aseg_REN_gmrois.nii.gz'
    RUNS  = sorted(glob.glob('/mnt/belegost/Projects/TASKFORCE/derivatives/global_proc/ses-*_task-mdtb_run*_final.nii.gz'))

    # cache key from the atlas filename (e.g. Ward would be 'ward-200')
    atlas_key = os.path.basename(ATLAS).replace('.nii.gz', '').replace('.nii', '')
    cache_path = os.path.join(WORKDIR, f'parcellated_{atlas_key}_{len(RUNS)}runs.pkl')

    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as fh:
            blob = pickle.load(fh)
        raw = blob['raw']
        assert blob['run_files'] == RUNS, "cached run list differs — delete the pkl to rebuild"
        print(f'loaded cached ROI timeseries: {cache_path}')
    else:
        from fastfuncstuff.cli.bsds import _load_volume
        from fastfuncstuff.dynamics.parcellate import parcellate_atlas
        atlas = _load_volume(ATLAS).astype(int)
        raw = []
        for f in RUNS:                                   # the slow part (volume load)
            _, ts = parcellate_atlas(_load_volume(f), atlas)   # -> (D, N)
            raw.append(ts)
        with open(cache_path, 'wb') as fh:
            pickle.dump({'raw': raw, 'run_files': RUNS, 'atlas': ATLAS}, fh)
        print(f'parcellated {len(raw)} runs -> cached {cache_path}')

    # preprocessing is cheap — always run fresh so you can tweak detrend/standardize
    sessions = preprocess_sessions(raw, detrend_degree=3, standardize=None)
    print(f'{len(sessions)} real sessions, D={sessions[0].shape[0]} ROIs, '
        f'N={[s.shape[1] for s in sessions]} TRs')


## MATLAB EXPORT
Lives here
/home/logan/local_code/Cai_Multiple_Demand_System_2023

In [ ]:
# Export to matlab
from fastfuncstuff.dynamics.bsds.export_matlab import write_matlab_bsds_input

# Ites here /home/logan/local_code/Cai_Multiple_Demand_System_2023)
m_script = write_matlab_bsds_input(
      sessions, '/mnt/belegost/Projects/TASKFORCE/derivatives/bsds/mdtb.mat',
      max_nstates=13, max_ldim=3,   # match whatever you pass to fit_bsds
)
print(m_script)  # edit BSDS_REPO at the top, then run it in MATLAB/Octave

## 2.1 Parameter Sweep with Xval

In [ ]:
# # choosing params
# from fastfuncstuff.dynamics.model_selection import grid_search_bsds

# results = grid_search_bsds(   
#     sessions,
#     n_states_grid=[8, 12, 24],
#     max_ldim_grid=[3, 5, 7],
#     n_folds=10,        # leave-10-runs-out groups rather than all 30 (cheaper)
#     n_init=5, n_init_iter=10, n_iter=60,   # coarse budget for the sweep
# )
# for r in results[:5]:
#     print(r.n_states, r.max_ldim, round(r.per_timepoint_loglik, 4))
#     from fastfuncstuff.dynamics import plots
#     plots.plot_selection_surface(results)   # heatmap of held-out LL/TR, best cell starred



In [ ]:
# for r in results:
#     print(r.n_states, r.max_ldim, round(r.per_timepoint_loglik, 4))

## 2.2 Fit BSDS

`fit_bsds` runs variational Bayes with several random restarts, keeps the best by free energy,
and decodes the most-likely state sequence. `n_states` is fixed and always fully returned — ARD
does **not** drop states. It only prunes each state's *latent factor* count (columns of the
loading matrix), which shows up as `model.effective_dim`. To see which of your `n_states` are
actually used, look at `stats.group_occupancy` — states the data never visits will simply have
~0 occupancy, not disappear from the output.

In [ ]:
# model = fit_bsds(sessions, n_states=13, max_ldim=5, n_init=50, n_iter=250, seed=0, tol=1e-8, kmeans_pca_dim=None)
# stats = compute_state_stats(model, tr=TR)  # TR in seconds -> lifetimes in seconds
# print('converged:', model.converged, 'in', len(model.objective_history), 'iters')
# print('occupancy:', np.round(stats.group_occupancy, 3))

In [ ]:
import os

from fastfuncstuff.dynamics.bsds.model import load_bsds_model, save_bsds_model

FIT = dict(n_states=13, max_ldim=3, n_init=10, criterion="weights", n_iter=600, seed=0, tol=1e-3, kmeans_pca_dim=None)
FORCE_REFIT = True   # set True to ignore the cache and refit

# cache key = fit params + a signature of the data (runs / ROIs / total TRs), so a
# different sessions list (new atlas, more runs) won't silently reuse an old fit.
sig   = f"{len(sessions)}runs_{sessions[0].shape[0]}roi_{sum(int(s.shape[1]) for s in sessions)}tr"
tag   = "_".join(f"{k}{v}" for k, v in FIT.items())
cache = os.path.join(WORKDIR, f"bsds_{tag}_{sig}.npz")

if os.path.exists(cache) and not FORCE_REFIT:
    model = load_bsds_model(cache)
    print(f"loaded cached fit: {os.path.basename(cache)}")
else:
    model = fit_bsds(sessions, **FIT)
    save_bsds_model(model, cache)
    print(f"fit + cached: {os.path.basename(cache)}")

stats = compute_state_stats(model, tr=TR)
print('converged:', model.converged, 'in', len(model.objective_history), 'iters')
print('occupancy:', np.round(stats.group_occupancy, 3))


## 2.1 Stupid GPU attempt

In [ ]:
TEST_GPU= True

if TEST_GPU:

    import torch
    dev = torch.device("cuda")   # or get_device("cuda")
    import os

    from fastfuncstuff.dynamics.bsds.model import load_bsds_model, save_bsds_model

    FIT = dict(n_states=13, max_ldim=3, n_init=100, criterion="weights", n_iter=600, seed=0, tol=1e-3, device=dev, kmeans_pca_dim=None)
    FORCE_REFIT = True   # set True to ignore the cache and refit

    # cache key = fit params + a signature of the data (runs / ROIs / total TRs), so a
    # different sessions list (new atlas, more runs) won't silently reuse an old fit.
    sig_gpu   = f"{len(sessions)}runs_gpu__{sessions[0].shape[0]}roi_{sum(int(s.shape[1]) for s in sessions)}tr"
    tag_gpu   = "_".join(f"{k}{v}" for k, v in FIT.items())
    cache_gpu = os.path.join(WORKDIR, f"bsds_{tag_gpu}_{sig_gpu}.npz")

    if os.path.exists(cache_gpu) and not FORCE_REFIT:
        model_gpu = load_bsds_model(cache_gpu)
        print(f"loaded cache_gpu fit: {os.path.basename(cache_gpu)}")
    else:
        model_gpu = fit_bsds(sessions, **FIT)
        save_bsds_model(model_gpu, cache_gpu)
        print(f"fit + cache_gpu: {os.path.basename(cache_gpu)}")

    stats_gpu = compute_state_stats(model_gpu, tr=TR)
    print('converged:', model_gpu.converged, 'in', len(model_gpu.objective_history), 'iters')
    print('occupancy:', np.round(stats_gpu.group_occupancy, 3))


In [ ]:
from fastfuncstuff.dynamics.model_selection import grid_search_bsds
import time; t=time.time()
res = grid_search_bsds(sessions, [9,13,17], [3,5,8], device="cuda", n_init=4, n_folds=3)
print(f"{time.time()-t:.1f}s"); [print(r.n_states, r.max_ldim, round(r.per_timepoint_loglik,4)) for r in res]


In [ ]:
from fastfuncstuff.dynamics.model_selection import grid_search_bsds
import time; t=time.time()
res = grid_search_bsds(sessions, [9,13,17], [3,5,8], device="cuda", n_init=4, n_folds=3, n_jobs = 3)
print(f"{time.time()-t:.1f}s"); [print(r.n_states, r.max_ldim, round(r.per_timepoint_loglik,4)) for r in res]


In [ ]:
res = grid_search_bsds(sessions, [9,13,17], [3,5,8], device="cuda", n_init=4, n_folds=3, n_jobs=3)


In [ ]:
GPU_PROFILE=True
if GPU_PROFILE:
    # import torch
    # with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CUDA,
    #                                         torch.profiler.ProfilerActivity.CPU]) as p:
    #     fit_bsds(sessions, n_states=13, max_ldim=3, n_init=1, n_iter=30, device="cuda")
    # print(p.key_averages().table(sort_by="cuda_time_total", row_limit=15))
    import time, torch
    from fastfuncstuff.dynamics.bsds.init import init_state
    from fastfuncstuff.dynamics.bsds import vb, hmm
    from fastfuncstuff.dynamics.bsds.elbo import lower_bound

    dev = torch.device("cuda")
    y = torch.cat([s.to(dev, torch.float64) for s in sessions], 1)
    lengths = [int(s.shape[1]) for s in sessions]
    st = init_state(y, lengths, n_states=13, ldim=3, seed=0, kmeans_pca_dim=None)

    # a couple real passes so intermediates are valid
    xi = g0 = None
    for _ in range(2):
        if xi is not None: vb.update_qtheta(st, xi, g0)
        vb.update_qnu(st); vb.update_qx(st, y); vb.update_ql(st, y)
        vb.update_psi(st, y); vb.update_mcl(st)
        lo = vb.compute_log_out_probs(st, [s.to(dev, torch.float64) for s in sessions])
        gm, xi, g0, ll = hmm.estep(lo, st.wa, st.wpi); st.qns = torch.cat(gm, 0)

    sess_gpu = [s.to(dev, torch.float64) for s in sessions]
    def t_ms(fn, n=20):
        torch.cuda.synchronize(); fn(); torch.cuda.synchronize()      # warm
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(n): fn()
        torch.cuda.synchronize(); return (time.perf_counter() - t0)/n*1000

    ops = {
    "compute_log_out_probs": lambda: vb.compute_log_out_probs(st, sess_gpu),
    "hmm.estep":  lambda: hmm.estep(vb.compute_log_out_probs(st, sess_gpu), st.wa, st.wpi),
    "update_qx":  lambda: vb.update_qx(st, y),
    "update_ql":  lambda: vb.update_ql(st, y),
    "update_psi": lambda: vb.update_psi(st, y),
    "lower_bound":lambda: lower_bound(st, y),
    }
    res = {k: t_ms(f) for k, f in ops.items()}
    res["hmm.estep(only)"] = res.pop("hmm.estep") - res["compute_log_out_probs"]
    for k, v in sorted(res.items(), key=lambda x: -x[1]):
        print(f"{k:24s} {v:8.2f} ms")



In [ ]:
import os, time, torch
from fastfuncstuff.dynamics.states import compute_state_stats

def timed_fit():
    torch.cuda.synchronize(); t = time.time()
    m = fit_bsds(sessions, n_states=13, max_ldim=3, n_init=10, n_iter=600,
                criterion="weights", tol=1e-3, device="cuda", seed=0, kmeans_pca_dim=20)
    torch.cuda.synchronize(); return m, time.time() - t

os.environ["FFS_BSDS_CUDAGRAPH"] = "0"; m_off, t_off = timed_fit()
os.environ["FFS_BSDS_CUDAGRAPH"] = "1"; m_on,  t_on  = timed_fit()

occ_off = compute_state_stats(m_off, tr=1.0).group_occupancy
occ_on  = compute_state_stats(m_on,  tr=1.0).group_occupancy
print(f"eager: {t_off:.1f}s   graph: {t_on:.1f}s   speedup: {t_off/t_on:.2f}x")
print("occupancy max |Δ|:", float(abs(occ_off - occ_on).max()))


In [ ]:
import time, torch
from fastfuncstuff.dynamics.bsds.init import init_state, _per_session_labels
y = torch.cat([s.to("cuda", torch.float64) for s in sessions], 1)
lengths = [int(s.shape[1]) for s in sessions]
def t_ms(fn, n=5):
    torch.cuda.synchronize(); fn(); torch.cuda.synchronize()
    torch.cuda.synchronize(); t=time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return (time.perf_counter()-t)/n*1000
print("full init_state   :", round(t_ms(lambda: init_state(y, lengths, 13, 3, seed=0, kmeans_pca_dim=20)),1), "ms")
print("  k-means labels  :", round(t_ms(lambda: _per_session_labels(y, lengths, 13, n_replicates=10, pca_dim=20, seed=0)),1), "ms")
print("  k-means (rep=3) :", round(t_ms(lambda: _per_session_labels(y, lengths, 13, n_replicates=3,  pca_dim=20, seed=0)),1), "ms")



In [ ]:
import time, torch
from fastfuncstuff.dynamics.bsds.init import init_state
y = torch.cat([s.to("cuda", torch.float64) for s in sessions], 1)
lengths = [int(s.shape[1]) for s in sessions]
def t_ms(fn, n=5):
    torch.cuda.synchronize(); fn(); torch.cuda.synchronize()
    torch.cuda.synchronize(); t=time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return (time.perf_counter()-t)/n*1000
print("init_state (batched):", round(t_ms(lambda: init_state(y, lengths, 13, 3, seed=0, kmeans_pca_dim=20)), 1), "ms")




In [ ]:
# That was 1823ms before — I'd expect it to drop to ~100-200ms (the ~10× launch collapse). Then a full fit to see the combined effect with the FB graph:

import time, torch
torch.cuda.synchronize(); t=time.time()
m = fit_bsds(sessions, n_states=13, max_ldim=3, n_init=10, n_iter=600,
            criterion="weights", tol=1e-3, device="cuda", seed=0)  # kmeans_pca_dim=20 default
torch.cuda.synchronize()
print(f"full fit: {time.time()-t:.1f}s")
from fastfuncstuff.dynamics.states import compute_state_stats
print("occupancy:", compute_state_stats(m, tr=1.0).group_occupancy.round(3))

In [ ]:
import torch
from fastfuncstuff.dynamics.bsds import hmm
# _orig = hmm.forward_backward_batched
# hmm.forward_backward_batched = torch.compile(_orig, mode="reduce-overhead")
# # warm it a few times (compile is slow on first calls), then re-run the t_ms(estep) block
# for _ in range(5):
#     hmm.estep(vb.compute_log_out_probs(st, sess_gpu), st.wa, st.wpi)
# print("estep after compile:", t_ms(lambda: hmm.estep(vb.compute_log_out_probs(st, sess_gpu), st.wa, st.wpi)) - t_ms(lambda: vb.compute_log_out_probs(st, sess_gpu)),
# "ms")
hmm.forward_backward_batched = _orig  # restore


In [ ]:
import torch, time
from collections import defaultdict
from fastfuncstuff.dynamics.bsds import hmm

log_a  = hmm.expected_log_transition(st.wa)
log_pi = hmm.expected_log_init(st.wpi)
lo     = vb.compute_log_out_probs(st, sess_gpu)            # list of (T,K), mixed T

groups = defaultdict(list)
for t in lo: groups[t.shape[0]].append(t)
T, grp = max(groups.items(), key=lambda kv: len(kv[1]))    # dominant length-group
stack  = torch.stack([t.to(torch.float64) for t in grp], 0)  # (B, T, K)
print(f"length-groups: {{T: n}} = { {k: len(v) for k,v in groups.items()} }; capturing T={T}, B={len(grp)}")

in_obs, in_a, in_pi = stack.clone(), log_a.clone(), log_pi.clone()

# warm up on a side stream (required before capture)
s = torch.cuda.Stream(); s.wait_stream(torch.cuda.current_stream())
with torch.cuda.stream(s):
    for _ in range(3):
        _ = hmm.forward_backward_batched(in_obs, in_a, in_pi)
torch.cuda.current_stream().wait_stream(s)

# capture the whole 2*(T-1)-step recursion into one graph
g = torch.cuda.CUDAGraph()
with torch.cuda.graph(g):
    out = hmm.forward_backward_batched(in_obs, in_a, in_pi)

def graph_fb():
    in_obs.copy_(stack); in_a.copy_(log_a); in_pi.copy_(log_pi)
    g.replay()

def t_ms(fn, n=30):
    torch.cuda.synchronize(); fn(); torch.cuda.synchronize()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return (time.perf_counter() - t0)/n*1000

print("eager forward_backward:", round(t_ms(lambda: hmm.forward_backward_batched(stack, log_a, log_pi)), 2), "ms")
print("graph forward_backward:", round(t_ms(graph_fb), 2), "ms")


## 3. Did it fit? — free energy

The variational free energy **F** must increase monotonically. A clean rising curve that flattens
means the fit converged; a jagged or still-climbing curve means raise `n_iter` or add restarts.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
plots.plot_convergence(model, ax)
plt.show()

In [ ]:
# GPU version
fig, ax = plt.subplots(figsize=(6, 3))
plots.plot_convergence(model_gpu, ax)
plt.show()

## 3.1 Do the states and LDIM look ok?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- ARD saturation QC: are states pinned at the ldim ceiling, or has ARD pruned? ---
eff   = model.effective_dim.numpy()          # (K,) active factors per state
ceil  = model.ldim                           # the max_ldim ceiling you set
K     = model.n_states

# occupancy straight from the MAP paths (no dependence on `stats`)
occ = np.zeros(K)
for v in model.viterbi_states:
    occ += np.bincount(v.numpy().astype(int), minlength=K)[:K]
occ /= occ.sum()

order     = np.argsort(-occ)                  # sort states by occupancy
occupied  = occ[order] > 1e-3
eff_o     = eff[order]
saturated = eff_o >= ceil                     # pinned at the ceiling = capacity-limited

# per-factor loading energy: a pruned factor's column shrinks to ~0 under ARD
energy = model.loadings.norm(dim=1).numpy()[order]   # (K, ldim) = ||Λ_s[:,j]||

# ---- verdict line ----
n_sat = int((saturated & occupied).sum())
print(f"max_ldim (ceiling) = {ceil}")
print(f"occupied states: {int(occupied.sum())}/{K}   "
    f"effective_dim over occupied: min={eff_o[occupied].min():.0f} "
    f"mean={eff_o[occupied].mean():.1f} max={eff_o[occupied].max():.0f}")
print(f"saturated (eff == ceiling) among occupied: {n_sat}"
    + ("  -> capacity-limited, try a higher max_ldim" if n_sat else
        "  -> ARD has slack, ceiling is not binding"))

# ---- figure ----
fig, (a0, a1) = plt.subplots(1, 2, figsize=(13, 4.5), width_ratios=[1.3, 1])
x = np.arange(K)
bars = a0.bar(x, eff_o, color=np.where(saturated, "#d64545", "#2a78d6"))
a0.bar(x[~occupied], eff_o[~occupied], color="#cccccc")   # grey out empty states
a0.axhline(ceil, ls="--", color="#52514e", lw=1.5, label=f"max_ldim = {ceil}")
a0.set_xticks(x, [f"S{s}\n{occ[s]:.2f}" for s in order], fontsize=6)
a0.set_ylabel("effective dim (ARD-active factors)")
a0.set_title("Per-state latent usage (red = saturated, grey = empty)")
a0.legend(frameon=False)

im = a1.imshow(np.log10(energy + 1e-8), aspect="auto", cmap="viridis")
a1.set_xlabel("latent factor"); a1.set_ylabel("state (occupancy order)")
a1.set_title("Loading energy  log₁₀‖Λ[:,j]‖   (dark = pruned)")
fig.colorbar(im, ax=a1, fraction=0.046, pad=0.04)
fig.suptitle(f"BSDS ARD saturation — K={K}, ceiling={ceil}", fontsize=13)
plt.tight_layout(); plt.show()


## 3.2 effective states?

In [ ]:
## How many states are *really* in play?
# exp(entropy(occupancy)) — weights a 0.001-occupancy state far less than a 0.3 one,
# so it's a better headline than the raw occupied count for "is it really 13 states?".
eff_k    = stats.effective_state_count
occupied = int((stats.group_occupancy > 1e-3).sum())
print(f"fit n_states       : {model.n_states}")
print(f"occupied (>0.001)  : {occupied}")
print(f"effective # states : {eff_k:.1f}")
print(f"  -> {eff_k:.1f} states carry the bulk of the time; the gap to {occupied} occupied "
    "is many low-occupancy states you can mostly ignore.")


## 3.3 reproducible repeats?

In [ ]:
WANT_REPRODUCE =True
if WANT_REPRODUCE:
    ## 3.2 Are the states reproducible across random inits?
    # Refit N times from different seeds, Hungarian-match each to a reference by FC pattern,
    # and score how consistently each state's connectivity comes back. Tall/tight bars =
    # trustworthy states; short or long-whiskered = init-dependent (candidate artifacts).
    from fastfuncstuff.dynamics.stability import state_stability

    stab = state_stability(
        sessions,
        n_states=model.n_states,
        max_ldim=model.ldim,
        n_repeats=4,                       # 4 independent fits (3 compared to the reference)
        n_init=10, n_iter=150, tol=1e-6,   # per-fit budget — this is N full fits, so trim for speed
        kmeans_pca_dim=None,               # match your main fit's (legacy) init
        seed=0, show_progress=True,
    )
    print(f"mean matched-FC over occupied states : {stab.mean_fc:.3f}   (1 = identical FC)")
    print(f"occupancy correlation (ref vs repeats): {stab.occupancy_correlation:.3f}")

    # per-state reproducibility, occupancy-ordered
    order = np.argsort(-stab.reference_occupancy)
    for s in order:
        if stab.reference_occupancy[s] > 1e-3:
            print(f"  S{s}: occ={stab.reference_occupancy[s]:.3f}  "
                f"mean FC={stab.per_state_fc[s]:.2f}  worst={stab.per_state_fc_min[s]:.2f}")

    plots.plot_state_stability(stab)   # bars: green=occupied, grey=empty, whisker=worst repeat


In [ ]:
plots.plot_state_stability(stab)

## 4. The core view — state probability time courses

This is the figure to read first. Each line is the posterior probability of one state over time.
Crisp, near-binary switching between well-separated states means the model found real structure;
mushy overlapping probabilities mean the states aren't well identified (ambiguous data, wrong
`n_states`, or too little data).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
plots.plot_state_timecourses(model, run_idx=2, ax=ax, tr=TR)
plt.show()

The MAP (Viterbi) path collapses that to a single most-likely state per frame — a compact ribbon:

In [ ]:
from fastfuncstuff.dynamics import plots

plots.plot_state_ribbons(model, tr=TR)   

In [ ]:
# # Lets plot this for a variable number of runs - so I set runs, and it makes rows with this plot, nice because each run is the same length
# # but lets be smart - they should be stacked as best as possible (some runs are slighly longer, but that )
# for run_idx in range(5):
#     fig, ax = plt.subplots(figsize=(11, 1.1))
#     plots.plot_state_ribbon(model, run_idx=run_idx, ax=ax, tr=TR)
#     plt.show()

## 4.1 Task Relationship?

In [ ]:
if USE_REAL_DATA:
    import glob
    from pathlib import Path

    from fastfuncstuff.design.bids_events import parse_bids_events
    from fastfuncstuff.dynamics import plots
    from fastfuncstuff.dynamics.task import align_states_to_task

    # One events.tsv per run, sorted the SAME way as your -input runs.
    events_dir = Path(
        "~/Dropbox/Areas/UM_fMRI_Task_Data/20251030_MDTB_TASKFORCE_TimingFiles/"
        "sub-pilot01_Remaking_Timing_Files/all_sessions_bids_events_simple"
    ).expanduser()

    event_files = sorted(str(p) for p in events_dir.glob("*_events.tsv"))
    print(f"Found {len(event_files)} event files in: {events_dir}")
    assert len(event_files) == len(model.responsibilities), \
        f"{len(event_files)} events vs {len(model.responsibilities)} runs — order/count must match"

    all_onsets, durations, condition_labels = parse_bids_events(
        event_files,
        event_ignore=None,          # e.g. ['rest','instruct'] to drop conditions
    )

    align = align_states_to_task(
        model, all_onsets, durations, condition_labels,
        tr=TR, hrf_delay=5.0,
    )
    print(f"{len(condition_labels)} conditions | NMI={align.normalized_mutual_info:.3f} "
        f"| mean purity={align.state_purity.mean():.2f}")

    # per-state: which condition dominates it
    for s, c in enumerate(align.dominant_condition):
        name = condition_labels[c] if c >= 0 else 'baseline'
        print(f"  S{s}: {name}  (purity {align.state_purity[s]:.2f})")

    p = plots.task_alignment_report(model, align, '/tmp/bsds_task_align.png', tr=TR)
    from IPython.display import Image; Image(p)


In [ ]:
if USE_REAL_DATA:
    import numpy as np

    from fastfuncstuff.design.bids_events import sort_bids_event_files
    from fastfuncstuff.dynamics.task import events_to_condition_labels

    # (a) What order did parse_bids_events actually use vs what you passed?
    print("you passed (first 4):", [Path(f).name for f in event_files[:4]])
    print("parser reordered to :", [p.name for p in sort_bids_event_files(event_files)[:4]])

    # (b) Run 0's parsed events — onsets should be spaced ~35s (5s instruct + 30s task)
    run = 0
    lens = [r.shape[0] for r in model.responsibilities]
    print(f"\nrun {run}: length {lens[run]} TR, "
        f"{sum(len(all_onsets[c][run]) for c in range(len(condition_labels)))} events")
    for cidx, name in enumerate(condition_labels):
        ons = np.asarray(all_onsets[cidx][run])
        if ons.size:
            print(f"  {name:22s} dur={durations[cidx]:5.1f}s  onsets={np.round(ons,1)}")

    # (c) The label strip as run-length blocks — this is exactly what you see as color
    labels = events_to_condition_labels(all_onsets, lens, tr=TR, hrf_delay=5.0, durations = durations)[run]
    change = np.concatenate([[True], labels[1:] != labels[:-1]])
    starts = np.flatnonzero(change)
    lengths = np.diff(np.concatenate([starts, [len(labels)]]))
    print(f"\nrun {run} label blocks (name : length):")
    for s, ln in zip(starts, lengths):
        v = labels[s]; nm = condition_labels[v] if v >= 0 else 'baseline'
        print(f"  t={s*TR:6.0f}s  {nm:22s} {ln} TR")


## 5. Switching dynamics — the transition matrix

Row *i*, column *j* is P(state *j* next | state *i* now). Brain states are **sticky**, so a good
fit has a dominant diagonal — here it should recover the ~0.96 stay probability we simulated.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
plots.plot_transition_matrix(model, ax)
plt.show()

## 6. How much, how long — occupancy & mean lifetime

Fractional **occupancy** is the share of time in each state; **mean lifetime** is the average
dwell duration (in seconds here, via the TR). The gray dots on occupancy are per-session values,
showing across-session variability. These per-state summaries are what the papers relate to
behaviour (e.g. occupancy of the optimal state predicts task performance).

In [ ]:
fig, (a0, a1) = plt.subplots(1, 2, figsize=(9, 3.2))
plots.plot_occupancy(stats, a0)
plots.plot_lifetime(stats, a1)
plt.tight_layout(); plt.show()

## 7. What defines each state — activation & functional connectivity

Each state has a **mean** (activation profile across ROIs) and a **covariance**. The covariance,
normalised to a correlation, is the state's **dynamic functional connectivity** — the network
pattern that state represents. This is the payoff of the factor-analysis observation model:
clean, denoised per-state FC.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
plots.plot_state_means(model, ax)
plt.show()

In [ ]:
plots.plot_state_fc(stats)
plt.show()

## 8. Everything at once — the QC panel

`qc_report` bundles all of the above into one figure — the same PNG `ffs_bsds` writes next to its
outputs. Glance at it to answer *did it fit, do the states make sense?*

In [ ]:
from IPython.display import Image

qc_path = '/tmp/bsds_demo_qc.png'
plots.qc_report(model, stats, qc_path, tr=TR)
Image(qc_path)

In [ ]:
# --- Cross-check the ffs_bsds fit against the reference MATLAB BSDS output ---
# `model` must be an ffs BSDSModel fit at the SAME n_states / max_ldim as the
# MATLAB export (26 / 10 for mdtb_model.mat) or the state-to-state match is
# meaningless. Two VB fits from different k-means inits land in different local
# optima, so this measures *structural* agreement, not identical numbers.
from fastfuncstuff.dynamics.bsds.matlab_compare import (
    compare_to_matlab,
    load_matlab_bsds,
    plot_comparison,
)

mat = load_matlab_bsds("/mnt/belegost/Projects/TASKFORCE/derivatives/bsds/mdtb_model_13_3_none.mat")
res = compare_to_matlab(model, mat)
print(f"occupied states (ffs vs MATLAB): {res.n_occupied_ffs} vs {res.n_occupied_matlab}")
print(f"mean matched FC       : {res.mean_matched_fc:.3f}")
print(f"occupancy correlation : {res.occupancy_correlation:.3f}")
print(f"transition correlation: {res.transition_correlation:.3f}")
print(f"lifetime correlation  : {res.lifetime_correlation:.3f}")
print(f"activation correlation: {res.activation_correlation:.3f}")
if res.temporal_frames:
    print(f"frame-wise MAP agreement: {res.temporal_agreement:.3f}  "
        f"(kappa={res.temporal_kappa:.3f}, over {res.temporal_frames} TRs)")
else:
    print("frame-wise MAP agreement: n/a (run count/lengths differ between fits)")

from IPython.display import Image

Image(plot_comparison(res, "/tmp/bsds_matlab_compare.png"))   # now has a scorecard panel



In [ ]:
from fastfuncstuff.dynamics.bsds.matlab_compare import temporal_diagnostics

diag = temporal_diagnostics(model, mat, res)
print(f"identity pairing   : {diag.identity_agreement:.3f}")
print(f"best run re-pairing: {diag.run_permuted_agreement:.3f}")
print(f"best frame shift   : {diag.best_shift} TR -> {diag.shift_agreement:.3f}")
print(f"per-run agreement  : {np.round(diag.per_run_agreement, 2)}")
if not np.array_equal(diag.run_permutation, np.arange(len(diag.run_permutation))):
    print("re-pairing reordered runs:", diag.run_permutation.tolist())

# and check lifetimes by value, not correlation:
from fastfuncstuff.dynamics.states import mean_lifetime

pooled = np.concatenate([v.numpy() for v in model.viterbi_states])
ffs_life = mean_lifetime(pooled, model.n_states, tr=1.0)
both = (res.ffs_occ > 1e-3) & (res.matlab_occ > 1e-3)
print("ffs    lifetimes:", np.round(ffs_life[res.ffs_state[both]], 1))
print("matlab lifetimes:", np.round(mat.lifetime[res.matlab_state[both]], 1))



In [ ]:
from IPython.display import Image

from fastfuncstuff.dynamics.bsds.matlab_compare import plot_map_comparison

Image(plot_map_comparison(model, mat, res, "/tmp/bsds_map_compare.png", tr=TR))


In [ ]:
from fastfuncstuff.dynamics.bsds.matlab_compare import temporal_diagnostics

diag = temporal_diagnostics(model, mat, res)
print(f"identity pairing   : {diag.identity_agreement:.3f}")
print(f"best run re-pairing: {diag.run_permuted_agreement:.3f}")
print(f"best frame shift   : {diag.best_shift} TR -> {diag.shift_agreement:.3f}")
print(f"per-run agreement  : {np.round(diag.per_run_agreement, 2)}")
if not np.array_equal(diag.run_permutation, np.arange(len(diag.run_permutation))):
    print("re-pairing reordered runs:", diag.run_permutation.tolist())

# and check lifetimes by value, not correlation:
from fastfuncstuff.dynamics.states import mean_lifetime

pooled = np.concatenate([v.numpy() for v in model.viterbi_states])
ffs_life = mean_lifetime(pooled, model.n_states, tr=1.0)
both = (res.ffs_occ > 1e-3) & (res.matlab_occ > 1e-3)
print("ffs    lifetimes:", np.round(ffs_life[res.ffs_state[both]], 1))
print("matlab lifetimes:", np.round(mat.lifetime[res.matlab_state[both]], 1))


## 9. Matching states across fits

Separate fits label their states arbitrarily. `match_states` aligns them — either by **state-space
closeness** (symmetric KL between the per-state Gaussians) or **temporal closeness** (correlation
of the probability time courses). This is how Cai et al. tracked one shared state across seven
different tasks.

In [ ]:
from fastfuncstuff.dynamics.matching import match_states

model_b = fit_bsds(sessions, n_states=26, max_ldim=6, n_init=5, n_iter=100, seed=7)
m = match_states(model, model_b, method='state_space')
print('fit-A state -> fit-B state:', dict(zip(m.row_ind.tolist(), m.col_ind.tolist())))

### Applying a fitted model to new data, and directed connectivity

`decode` applies a **fitted** model to new runs with parameters fixed (inference only, no
refit) — for held-out runs, or to line two fits up on the same time axis for temporal matching.
And beyond the undirected state covariance (FC above), `per_state_directed_connectivity` fits a
VAR on the ROI data per state, giving *directed* effective connectivity (`B[s][i, j]` = ROI j →
ROI i within state s).

In [ ]:
from fastfuncstuff.dynamics.bsds.model import decode
from fastfuncstuff.dynamics.connectivity import per_state_directed_connectivity

held_out, _, _ = simulate(n_sessions=1, seed=99)   # a run the model never saw
dec = decode(model, held_out)
occ = np.bincount(dec.viterbi_states[0].numpy(), minlength=model.n_states) / held_out[0].shape[1]
print('decoded held-out run:', dec.viterbi_states[0].shape[0], 'frames | occupancy', np.round(occ, 2))

B, _ = per_state_directed_connectivity(model, sessions)
print('directed connectivity:', tuple(B.shape), '(K, D, D)')

## 9c. Network metrics — how integrated is each state?

Each state's FC is a weighted graph. Graph-theoretic measures ([Taghia 2018] Fig. 6) summarise
them: **global efficiency** (integration — how short the paths are), **clustering** (segregation
— cliquishness), and per-node **strength**. A state high on efficiency is an integrated,
globally-coupled regime; a high-clustering state is modular.

In [ ]:
from fastfuncstuff.dynamics.graph import state_graph_metrics

gm = state_graph_metrics(stats.state_fc)
plots.plot_graph_metrics(gm)
plt.show()

## 9d. Switching dynamics — rate, bursts, and paths

Beyond the transition *matrix*: how **often** the brain switches, **when** (bursts of instability
vs stable epochs, via a windowed switch rate), and which multi-step **paths** between states
recur ([Cai 2024] Fig. 4f/5). Switch paths collapse dwell time — they're about *order* of visits.

In [ ]:
from fastfuncstuff.dynamics.switching import compute_switch_stats

ss = compute_switch_stats(model, tr=TR)
print('group switch rate:', round(ss.group_switch_rate, 3),
      '| per minute:', round(ss.switch_rate_per_minute, 1))
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3), width_ratios=[1.6, 1])
plots.plot_switch_rate_over_time(model, a0, tr=TR)
plots.plot_switch_paths(ss, a1)
plt.tight_layout(); plt.show()

## 9e. Brain–behaviour — does state dynamics predict behaviour?

The payoff analysis ([Cai 2024]: occupancy of the optimal state predicts performance). This needs
**many sessions** and a **behaviour** value per session — the *serious-analysis* tier. Everything
is **leave-one-session-out cross-validated**, because with tens of sessions in-sample fit is
meaningless. Here we simulate a 30-run cohort and plant a behaviour driven by state occupancy, so
you can see the held-out model recover it (with your data, drop in your real behaviour vector).

In [ ]:
from fastfuncstuff.dynamics.behavior import (
    canonical_correlation,
    cross_validated_prediction,
    session_feature_matrix,
)

cohort, _, _ = simulate(n_sessions=30, t=200, seed=3)
cohort_model = fit_bsds(cohort, n_states=4, max_ldim=4, n_init=3, n_iter=80, seed=0)
cohort_stats = compute_state_stats(cohort_model, tr=TR)
X, names = session_feature_matrix(cohort_stats)          # (S, 2K): occupancy + lifetime
occ = cohort_stats.subject_occupancy
behaviour = 3.0 * occ[:, 0] - 2.0 * occ[:, 1] + 0.05 * np.random.default_rng(1).standard_normal(len(occ))

pred = cross_validated_prediction(X, behaviour, alpha=1.0, feature_names=names)
print('held-out R2:', round(pred.r2, 2), '| r:', round(pred.correlation, 2))
fig, ax = plt.subplots(figsize=(4, 4))
plots.plot_prediction(pred, ax)
plt.show()

**Task decoding & CCA** use the same feature matrix. `loso_classification` decodes a categorical
condition (here: runs with fast vs slow switching), and `canonical_correlation` relates the state
features to a *multivariate* behaviour battery — both cross-validated / whitened for small S.

In [ ]:
from fastfuncstuff.dynamics.behavior import loso_classification, session_feature_matrix

# two conditions that differ only in switching speed (stay probability):
fast, _, _ = simulate(n_sessions=12, t=200, stay=0.85, seed=10)
slow, _, _ = simulate(n_sessions=12, t=200, stay=0.97, seed=20)
two_model = fit_bsds(fast + slow, n_states=4, max_ldim=4, n_init=3, n_iter=80, seed=0)
two_stats = compute_state_stats(two_model, tr=TR)
two_switch = compute_switch_stats(two_model, tr=TR)
Xc, _ = session_feature_matrix(two_stats, switch_stats=two_switch,
                               include=('occupancy', 'lifetime', 'switch_rate'))
labels = np.array([0] * 12 + [1] * 12)
clf = loso_classification(Xc, labels)
print('task-decoding accuracy (held-out):', round(clf.accuracy, 2))

rng = np.random.default_rng(7)
reaction_time = 2.0 * occ[:, 2] + rng.standard_normal(len(occ))   # noisy, partly state-linked
Y = np.column_stack([behaviour, reaction_time])
cca = canonical_correlation(X, Y, n_components=2)
print('canonical correlations:', np.round(cca.correlations, 2))

## 10. The CEBRA bridge

BSDS (discrete states, temporal structure) and [CEBRA](https://cebra.ai) (continuous nonlinear
manifold) decompose *orthogonal axes* of the same dynamics. `export.py` prepares the hand-off
(no CEBRA dependency): concatenate sessions into the `(N, D)` matrix CEBRA wants, and grab the
frame-aligned state labels to **colour a CEBRA embedding**. If the states occupy distinct
territories on the manifold, that's convergent evidence they're real.

In [ ]:
from fastfuncstuff.dynamics.export import frame_aligned_labels, prepare_cebra_inputs

X, lengths = prepare_cebra_inputs(sessions)
labels, _ = frame_aligned_labels(model)
print('CEBRA input:', X.shape, '| labels:', labels.shape, '| session lengths:', lengths)
print('# then: emb = CEBRA().fit_transform(X); colour emb by `labels`;')
print('#       state_embedding_separation(emb, labels) scores how distinct they are.')

## Using real data

Replace `simulate()` with your own sessions — each a `(D, N)` ROI time series (`torch`/`numpy`).
Get there from 4-D fMRI with `fastfuncstuff.dynamics.parcellate` (atlas, or data-driven
contiguity-constrained Ward for a precision individual), and `preprocess_sessions` for per-run
detrend + z-score. Keep `D` in the tens–low-hundreds.

The whole pipeline is also one CLI call:

```bash
ffs_bsds -input run*.1D -prefix out/sub01 -n_states 6 -tr 0.72 -plots all
# or from volumes:
ffs_bsds -input run*.nii.gz -parcellation atlas -atlas schaefer.nii.gz \
         -prefix out/sub01 -tr 0.72 -plots all
```

which writes the model bundle, per-run state time courses, and these same figures.

# MatLab Comparison guide 

1. What to comment out in MATLAB (the AR inertness test)

In /home/logan/local_code/Cai_Multiple_Demand_System_2023/BSDS/functions/learnAR_FA.m, comment out the single line:

inferAR3;

Leave vbhafa.m untouched — that keeps the one-time iteration-1 AR effect identical, so you're isolating exactly the main-loop AR contribution.
Rerun mdtb_run with the same .mat. Prediction: the free-energy trajectory and occupancy come out identical to the stock run, proving AR is inert
(overwritten by inferQX before anything reads it). If they differ, I'm wrong and want to know.

2. Saving MATLAB outputs for comparison

Your generated _run.m already does save('..._model.mat', 'model', '-v7.3') — but -v7.3 is HDF5, which scipy.io.loadmat can't read. Easiest fix:
add these lines to the end of your .m so the key numbers land in plain text (no mat73 dependency needed):
```
writematrix(model.fractional_occupancy_group_wise, 'mdtb_matlab_occupancy.csv');
writematrix(net_opt_Fhist_or_LL, 'mdtb_matlab_Fhist.csv');   % model.net.Fhist
% per-state covariances (dynamic FC), one CSV each:
for s = 1:numel(model.estimated_covariance)
    writematrix(model.estimated_covariance{s}, sprintf('mdtb_matlab_fc_state%02d.csv', s));
end
```

Then in Python: np.loadtxt('mdtb_matlab_occupancy.csv', delimiter=',') and compare to stats.group_occupancy. (For the AR test in #1, just eyeball
the printed fractional_occupancy_group_wise and final Fhist between the two runs — you don't even need the CSVs.)

3. Yes — MATLAB will find a different answer, and that's fine for now

Correct. Two independent sources of divergence, both expected and not bugs:
- k-means init: their initPoteriors and our init.py seed from different random draws → different starting partition.
- Non-convex VB: switching FA has many local optima; different starts land in different ones.

So don't expect matching labels or identical occupancy. The right comparison is structural: match states (Hungarian on the per-state covariances)
and check the FC patterns and occupancy spread agree, and compare final free energy F — if both reach similar F, they're equally-good optima. For
your exhaustive testing, that's the bar: "comparably good and structurally similar," not "bit-identical." If one collapses and the other doesn't,
that's the signal worth chasing.

One reminder for your testing: export with the same max_ldim you actually use (5, not 10) — otherwise you're comparing different models and paying
MATLAB's ~ldim⁴ AR cost for nothing.
